We have a new park id and general layout, so want to create a new workflow that uses these.

Additionally, I've been returning the visibility outputs as hexes, but this massively increases the daafile size (the polygon has multiple points, vs. a single point value)

Also, instead of running the entire England dataframe in one task array (with a limit of four concurrent task array jobs), I want to run it in a few sections.

But first I need to figure out the new park names.

In [1]:
import pandas as pd
import os
import glob


In [2]:
old_eng_filenames = pd.read_csv("england_filenames.csv")
old_wales_filenames = pd.read_csv("wales_filenames.csv")

In [3]:
# build a list of new filenames based on contents of 'parks_final' folder
folders = glob.glob("parks_final/*")


In [4]:
len(folders)

592

In [5]:
# drop anything in folder that contains pocket_parks
folders = [f for f in folders if "pocket_parks" not in f]

In [6]:
len(folders)

296

In [7]:
# cut parks_final/ out of the folder names
folders = [f.replace("parks_final/", "") for f in folders]

In [8]:
folder_df = pd.DataFrame(folders, columns=["folder_name"])

In [9]:
# drop _parks.geojson from the folder names to create a new column for placename
folder_df["placename"] = folder_df["folder_name"].apply(lambda x: x.replace("_parks.geojson", ""))

In [10]:
folder_df

,folder_name,placename
0,"Kingston upon Hull, City of_parks.geojson","Kingston upon Hull, City of"
1,Cornwall_parks.geojson,Cornwall
2,Newcastle upon Tyne_parks.geojson,Newcastle upon Tyne
3,Rother_parks.geojson,Rother
4,Rochford_parks.geojson,Rochford
...,...,...
291,Gwynedd_parks.geojson,Gwynedd
292,Worcester_parks.geojson,Worcester
293,Stockton-on-Tees_parks.geojson,Stockton-on-Tees
294,Vale of White Horse_parks.geojson,Vale of White Horse


In [11]:
print(old_wales_filenames)

# drop _parks.geojson from the folder names to create a new column for placename
old_wales_filenames["placename"] = old_wales_filenames["filename"].apply(lambda x: x.replace("_pp_or_g_cmb.geojson", ""))

print(old_wales_filenames)

                                 filename
0             Swansea_pp_or_g_cmb.geojson
1   Rhondda Cynon Taf_pp_or_g_cmb.geojson
2        Denbighshire_pp_or_g_cmb.geojson
3             Torfaen_pp_or_g_cmb.geojson
4   Vale of Glamorgan_pp_or_g_cmb.geojson
5               Powys_pp_or_g_cmb.geojson
6     Carmarthenshire_pp_or_g_cmb.geojson
7            Bridgend_pp_or_g_cmb.geojson
8       Blaenau Gwent_pp_or_g_cmb.geojson
9             Cardiff_pp_or_g_cmb.geojson
10         Ceredigion_pp_or_g_cmb.geojson
11              Conwy_pp_or_g_cmb.geojson
12      Pembrokeshire_pp_or_g_cmb.geojson
13         Flintshire_pp_or_g_cmb.geojson
14     Merthyr Tydfil_pp_or_g_cmb.geojson
15            Newport_pp_or_g_cmb.geojson
16            Gwynedd_pp_or_g_cmb.geojson
17  Neath Port Talbot_pp_or_g_cmb.geojson
18            Wrexham_pp_or_g_cmb.geojson
19         Caerphilly_pp_or_g_cmb.geojson
20      Monmouthshire_pp_or_g_cmb.geojson
21   Isle of Anglesey_pp_or_g_cmb.geojson
                                 f

In [12]:
print(old_eng_filenames)

# drop _parks.geojson from the folder names to create a new column for placename
old_eng_filenames["placename"] = old_eng_filenames["filename"].apply(lambda x: x.replace("_pp_or_g_cmb.geojson", ""))

print(old_eng_filenames)

                                     filename
0              Hartlepool_pp_or_g_cmb.geojson
1           Middlesbrough_pp_or_g_cmb.geojson
2    Redcar and Cleveland_pp_or_g_cmb.geojson
3        Stockton-on-Tees_pp_or_g_cmb.geojson
4              Darlington_pp_or_g_cmb.geojson
..                                        ...
291              Trafford_pp_or_g_cmb.geojson
292                 Wigan_pp_or_g_cmb.geojson
293              Knowsley_pp_or_g_cmb.geojson
294             Liverpool_pp_or_g_cmb.geojson
295             St Helens_pp_or_g_cmb.geojson

[296 rows x 1 columns]
                                     filename             placename
0              Hartlepool_pp_or_g_cmb.geojson            Hartlepool
1           Middlesbrough_pp_or_g_cmb.geojson         Middlesbrough
2    Redcar and Cleveland_pp_or_g_cmb.geojson  Redcar and Cleveland
3        Stockton-on-Tees_pp_or_g_cmb.geojson      Stockton-on-Tees
4              Darlington_pp_or_g_cmb.geojson            Darlington
..              

In [24]:
# based on the folder names, create a new column for the country
# check if placename is in the old_eng_filenames or old_wales_filenames, and assign the country accordingly
folder_df["country"] = folder_df["placename"].apply(lambda x: "England" if x in old_eng_filenames["placename"].values else "Wales" if x in old_wales_filenames["placename"].values else "NONE")

In [27]:
folder_df.country.unique()

<StringArray>
['England', 'Wales']
Length: 2, dtype: str

In [33]:
wales = folder_df.where(folder_df["country"] == "Wales").dropna()

In [35]:
wales.folder_name.to_csv("parameter_files/wales_filenames_00.csv", index=False, header=False)

In [42]:
england = folder_df.where(folder_df["country"] == "England").dropna()

In [43]:
len(england)

275

From https://www.geeksforgeeks.org/python/how-to-create-multiple-csv-files-from-existing-csv-file-using-pandas/, how to split a pandas dataframe into multiple csvs

In [44]:
# # (55 * 5)

# # no of csv files with row size
# k = 2
# size = 5

# # read DataFrame
# data = pd.read_csv("Customers.csv")

# for i in range(k):
#     df = data[size*i:size*(i+1)]
#     df.to_csv(f'Customers_{i+1}.csv', index=False)

# df_1 = pd.read_csv("Customers_1.csv")
# print(df_1)

# df_2 = pd.read_csv("Customers_2.csv")
# print(df_2)

# no of csv files with row size
k = 5
size = 55

for i in range(k):
    df = england[size*i:size*(i+1)]
    df.folder_name.to_csv(f"parameter_files/eng_filenames_0{i+1}.csv", index=False, header=False)



